In [1]:
!pip install requests

No global/local python version has been set yet. Please set the global/local version by typing:
pyenv global 3.7.4
pyenv local 3.7.4


In [2]:
import re
import time
import unicodedata
from pathlib import Path

import requests

In [3]:
coordonnees = {
    "Albigny-sur-Saône": (45.874994, 4.833002),
    "Bron": (45.733925, 4.912376),
    "Caluire-et-Cuire": (45.788408, 4.840143),
    "Champagne-au-Mont-d'Or": (45.794851, 4.792018),
    "Chassieu": (45.738192, 4.969525),
    "Collonges-au-Mont-d'Or": (45.826654, 4.843424),
    "Couzon-au-Mont-d'Or": (45.845998, 4.832412),
    "Décines-Charpieu": (45.771873, 4.955767),
    "Fontaines-sur-Saône": (45.833936, 4.847240),
    "La Mulatière": (45.727369, 4.814635),

    "Lyon 1er Arrondissement": (45.768952, 4.830784),
    "Lyon 2e Arrondissement": (45.752383, 4.828261),
    "Lyon 3e Arrondissement": (45.754844, 4.863078),
    "Lyon 4e Arrondissement": (45.777987, 4.825779),
    "Lyon 5e Arrondissement": (45.758478, 4.809764),
    "Lyon 6e Arrondissement": (45.770945, 4.851613),
    "Lyon 7e Arrondissement": (45.741779, 4.840139),
    "Lyon 8e Arrondissement": (45.737008, 4.869250),
    "Lyon 9e Arrondissement": (45.779867, 4.807047),

    "Neuville-sur-Saône": (45.873743, 4.839111),
    "Oullins": (45.717045, 4.806313),
    "Pierre-Bénite": (45.703256, 4.821200),
    "Rillieux-la-Pape": (45.810883, 4.886695),
    "Saint-Cyr-au-Mont-d'Or": (45.799791, 4.819602),
    "Saint-Didier-au-Mont-d'Or": (45.791574, 4.808535),
    "Saint-Fons": (45.709265, 4.857221),
    "Saint-Genis-Laval": (45.699345, 4.799343),
    "Saint-Germain-au-Mont-d'Or": (45.887543, 4.803977),
    "Saint-Priest": (45.710196, 4.916000),
    "Sainte-Foy-lès-Lyon": (45.749565, 4.800896),
    "Tassin-la-Demi-Lune": (45.763256, 4.778610),
    "Vaulx-en-Velin": (45.771718, 4.921055),
    "Villeurbanne": (45.769355, 4.884227),
    "Vénissieux": (45.709458, 4.872295),
    "Écully": (45.775089, 4.778673),
}

In [4]:
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

variables_meteo = (
    "temperature_2m,"
    "relative_humidity_2m,"
    "apparent_temperature,"
    "precipitation,"
    "rain,"
    "snowfall,"
    "weather_code,"
    "wind_speed_10m,"
    "wind_gusts_10m,"
    "is_day,"
    "visibility"
)

In [5]:
dossier_csv = Path("data1/weather_raw")

dossier_csv.mkdir(
    parents=True,
    exist_ok=True,
)

In [6]:
def normaliser_nom(nom):
    nom = unicodedata.normalize("NFKD", nom)
    nom = nom.encode("ascii", "ignore").decode("ascii")
    nom = nom.lower()
    nom = re.sub(r"[^a-z0-9]+", "_", nom)

    return nom.strip("_")

In [7]:
print(normaliser_nom("Décines-Charpieu"))
print(normaliser_nom("Lyon 1er Arrondissement"))

decines_charpieu
lyon_1er_arrondissement


In [8]:
#Fonction de téléchargement CSV

def telecharger_meteo(
    commune,
    latitude,
    longitude,
):
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": "2023-01-01",
        "end_date": "2026-08-30",
        "minutely_15": variables_meteo,
        "timezone": "Europe/Paris",
        "format": "csv",
    }

    print(f"Téléchargement : {commune}")

    response = requests.get(
        url,
        params=params,
        timeout=300,
    )

    response.raise_for_status()

    return response.text

In [9]:
# Sauvegarder le CSV

def sauvegarder_csv(
    commune,
    contenu_csv,
):
    nom_fichier = (
        f"{normaliser_nom(commune)}.csv"
    )

    chemin = dossier_csv / nom_fichier

    with chemin.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as fichier:
        fichier.write(contenu_csv)

    print(f"✅ Fichier créé : {chemin}")

In [11]:
# Test d'abord avec Villeurbanne
commune = "Villeurbanne"

latitude, longitude = coordonnees[commune]

contenu_csv = telecharger_meteo(
    commune,
    latitude,
    longitude,
)

print(contenu_csv[:1000])

Téléchargement : Villeurbanne
latitude,longitude,elevation,utc_offset_seconds,timezone,timezone_abbreviation
45.760002,4.8799996,173.0,7200,Europe/Paris,GMT+2

time,temperature_2m (°C),relative_humidity_2m (%),apparent_temperature (°C),precipitation (mm),rain (mm),snowfall (cm),weather_code (wmo code),wind_speed_10m (km/h),wind_gusts_10m (km/h),is_day (),visibility (m)
2023-01-01T00:00,15.2,67,11.0,0.00,0.00,0.00,0,27.1,59.0,0,24140.00
2023-01-01T00:15,15.1,67,11.0,0.00,0.00,0.00,0,26.8,59.0,0,24140.00
2023-01-01T00:30,15.1,68,11.1,0.00,0.00,0.00,0,26.1,59.0,0,24140.00
2023-01-01T00:45,15.0,68,11.1,0.00,0.00,0.00,0,25.9,58.7,0,24140.00
2023-01-01T01:00,15.0,68,11.0,0.00,0.00,0.00,0,25.5,58.3,0,24140.00
2023-01-01T01:15,15.0,68,11.1,0.00,0.00,0.00,0,25.4,57.6,0,24140.00
2023-01-01T01:30,15.0,67,11.0,0.00,0.00,0.00,0,25.3,56.5,0,24140.00
2023-01-01T01:45,15.0,67,11.0,0.00,0.00,0.00,0,25.2,55.4,0,24140.00
2023-01-01T02:00,14.9,67,10.9,0.00,0.00,0.00,0,25.5,54.7,0,24140.00
2023-01-01T02:15

In [12]:
sauvegarder_csv(
    commune,
    contenu_csv,
)

✅ Fichier créé : data1\weather_raw\villeurbanne.csv


In [14]:
# Vérifier le nombre de lignes 
with open(
    "data1/weather_raw/villeurbanne.csv",
    encoding="utf-8",
) as fichier:
    lignes = fichier.readlines()

print("Nombre total de lignes :", len(lignes))

Nombre total de lignes : 128452


In [15]:
# Télécharger toutes les communes
for index, (
    commune,
    (latitude, longitude),
) in enumerate(
    coordonnees.items(),
    start=1,
):
    print()
    print(
        f"[{index}/{len(coordonnees)}] "
        f"{commune}"
    )

    chemin = (
        dossier_csv
        / f"{normaliser_nom(commune)}.csv"
    )

    # Ne pas retélécharger un fichier déjà présent.
    if chemin.exists():
        print("⏭️ Fichier déjà présent")
        continue

    try:
        contenu_csv = telecharger_meteo(
            commune,
            latitude,
            longitude,
        )

        sauvegarder_csv(
            commune,
            contenu_csv,
        )

    except requests.RequestException as error:
        print(
            f"❌ Erreur pour {commune} : "
            f"{error}"
        )

    # Évite d'enchaîner trop rapidement
    # les grosses requêtes.
    time.sleep(15)


[1/35] Albigny-sur-Saône
Téléchargement : Albigny-sur-Saône
✅ Fichier créé : data1\weather_raw\albigny_sur_saone.csv

[2/35] Bron
Téléchargement : Bron
✅ Fichier créé : data1\weather_raw\bron.csv

[3/35] Caluire-et-Cuire
Téléchargement : Caluire-et-Cuire
✅ Fichier créé : data1\weather_raw\caluire_et_cuire.csv

[4/35] Champagne-au-Mont-d'Or
Téléchargement : Champagne-au-Mont-d'Or
✅ Fichier créé : data1\weather_raw\champagne_au_mont_d_or.csv

[5/35] Chassieu
Téléchargement : Chassieu
✅ Fichier créé : data1\weather_raw\chassieu.csv

[6/35] Collonges-au-Mont-d'Or
Téléchargement : Collonges-au-Mont-d'Or
✅ Fichier créé : data1\weather_raw\collonges_au_mont_d_or.csv

[7/35] Couzon-au-Mont-d'Or
Téléchargement : Couzon-au-Mont-d'Or
✅ Fichier créé : data1\weather_raw\couzon_au_mont_d_or.csv

[8/35] Décines-Charpieu
Téléchargement : Décines-Charpieu
✅ Fichier créé : data1\weather_raw\decines_charpieu.csv

[9/35] Fontaines-sur-Saône
Téléchargement : Fontaines-sur-Saône
✅ Fichier créé : data1\weat

In [16]:
import csv
from datetime import datetime

In [17]:
def ligne_vers_document(
    ligne,
    commune,
    latitude,
    longitude,
):
    return {
        "commune": commune,
        "location": {
            "type": "Point",
            "coordinates": [
                longitude,
                latitude,
            ],
        },
        "datetime": datetime.fromisoformat(
            ligne["time"]
        ),
        "temperature_2m_c": float(
            ligne["temperature_2m (°C)"]
        ),
        "relative_humidity_2m_pct": int(
            ligne["relative_humidity_2m (%)"]
        ),
        "apparent_temperature_c": float(
            ligne["apparent_temperature (°C)"]
        ),
        "precipitation_mm": float(
            ligne["precipitation (mm)"]
        ),
        "rain_mm": float(
            ligne["rain (mm)"]
        ),
        "snowfall_cm": float(
            ligne["snowfall (cm)"]
        ),
        "weather_code": int(
            ligne["weather_code (wmo code)"]
        ),
        "wind_speed_10m_kmh": float(
            ligne["wind_speed_10m (km/h)"]
        ),
        "wind_gusts_10m_kmh": float(
            ligne["wind_gusts_10m (km/h)"]
        ),
        "is_day": bool(
            int(ligne["is_day ()"])
        ),
        "visibility_m": float(
            ligne["visibility (m)"]
        ),
    }

In [ ]:
#fonction pour repérer le début réel des données dans le CSV Open-Meteo :

def trouver_debut_donnees(chemin):
    with chemin.open(
        "r",
        encoding="utf-8",
    ) as fichier:
        for numero_ligne, ligne in enumerate(fichier):
            if ligne.startswith("time,"):
                return numero_ligne

    raise ValueError(
        f"En-tête météo introuvable dans {chemin}"
    )

In [ ]:
#fonction qui transforme tout un fichier CSV en documents Python :

def lire_documents_csv(
    commune,
    latitude,
    longitude,
    chemin_csv,
):
    debut_donnees = trouver_debut_donnees(
        chemin_csv
    )

    documents = []

    with chemin_csv.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as fichier:

        for _ in range(debut_donnees):
            next(fichier)

        reader = csv.DictReader(fichier)

        for ligne in reader:
            document = ligne_vers_document(
                ligne,
                commune,
                latitude,
                longitude,
            )

            documents.append(document)

    return documents

In [21]:
# tester tous les fichiers CSV

for index, (
    commune,
    (latitude, longitude),
) in enumerate(
    coordonnees.items(),
    start=1,
):
    print()
    print(
        f"[{index}/{len(coordonnees)}] "
        f"{commune}"
    )

    chemin = (
        dossier_csv
        / f"{normaliser_nom(commune)}.csv"
    )

    if not chemin.exists():
        print(f"❌ Fichier absent : {chemin}")
        continue

    try:
        documents = lire_documents_csv(
            commune,
            latitude,
            longitude,
            chemin,
        )

        print(
            f"✅ Nombre de documents : "
            f"{len(documents)}"
        )

        print("Premier document :")
        print(documents[0])

        print("Dernier document :")
        print(documents[-1])

    except Exception as error:
        print(
            f"❌ Erreur pour {commune} : "
            f"{error}"
        )


[1/35] Albigny-sur-Saône
✅ Nombre de documents : 128448
Premier document :
{'commune': 'Albigny-sur-Saône', 'location': {'type': 'Point', 'coordinates': [4.833002, 45.874994]}, 'datetime': datetime.datetime(2023, 1, 1, 0, 0), 'temperature_2m_c': 14.5, 'relative_humidity_2m_pct': 72, 'apparent_temperature_c': 10.8, 'precipitation_mm': 0.0, 'rain_mm': 0.0, 'snowfall_cm': 0.0, 'weather_code': 1, 'wind_speed_10m_kmh': 24.1, 'wind_gusts_10m_kmh': 54.0, 'is_day': False, 'visibility_m': 24140.0}
Dernier document :
{'commune': 'Albigny-sur-Saône', 'location': {'type': 'Point', 'coordinates': [4.833002, 45.874994]}, 'datetime': datetime.datetime(2026, 8, 30, 23, 45), 'temperature_2m_c': 22.4, 'relative_humidity_2m_pct': 76, 'apparent_temperature_c': 23.8, 'precipitation_mm': 0.0, 'rain_mm': 0.0, 'snowfall_cm': 0.0, 'weather_code': 2, 'wind_speed_10m_kmh': 10.3, 'wind_gusts_10m_kmh': 22.0, 'is_day': False, 'visibility_m': 34280.0}

[2/35] Bron
✅ Nombre de documents : 128448
Premier document :
{

In [ ]:
total_global = 0

for commune, (
    latitude,
    longitude,
) in coordonnees.items():

    chemin = (
        dossier_csv
        / f"{normaliser_nom(commune)}.csv"
    )

    if not chemin.exists():
        print(f"❌ {commune} : fichier absent")
        continue

    total, premiers, dernier = tester_csv(
        commune,
        latitude,
        longitude,
        chemin,
    )

    total_global += total

    print(
        f"{commune} : {total} documents"
    )

print()
print(
    "Nombre total de documents "
    "pour toutes les communes :",
    total_global,
)